In [ ]:
import pandas as pd
import numpy as np
import requests
from shapely.geometry import shape
from datetime import datetime

AMES_URL = "https://raw.githubusercontent.com/Padre-Media/dataset/main/Ames.csv"
ARCGIS_URL = "https://apps.storycounty.com/arcgis/rest/services/parcels/MapServer/0/query"

TRAIN_IN = r"C:\Users\Admin\Downloads\train_with_PID.csv"
TEST_IN  = r"C:\Users\Admin\Downloads\test_with_PID.csv"

def pid_to_geocode10(pid):
    if pd.isna(pid): 
        return None
    return str(int(pid)).zfill(10)

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

def fetch_centroids_by_geocode(geocodes, chunk_size=200):
    rows = []
    for batch in chunks(geocodes, chunk_size):
        where = "GEOCODE IN (" + ",".join([f"'{g}'" for g in batch]) + ")"
        params = {
            "where": where,
            "outFields": "GEOCODE",
            "returnGeometry": "true",
            "outSR": "4326",
            "f": "geojson",
            "resultRecordCount": 2000,
            "resultOffset": 0
        }

        r = requests.get(ARCGIS_URL, params=params, timeout=60)
        r.raise_for_status()
        gj = r.json()

        for feat in gj.get("features", []):
            g = feat["properties"]["GEOCODE"]
            geom = shape(feat["geometry"])
            c = geom.centroid
            rows.append({"GEOCODE": g, "Latitude": c.y, "Longitude": c.x})

    return pd.DataFrame(rows).drop_duplicates("GEOCODE")

train = pd.read_csv(TRAIN_IN)
test  = pd.read_csv(TEST_IN)

ames = pd.read_csv(AMES_URL, usecols=["PID","Latitude","Longitude"]).drop_duplicates("PID")

for df in [train, test, ames]:
    df["PID"] = pd.to_numeric(df["PID"], errors="coerce").astype("Int64")

train_geo = train.merge(ames, on="PID", how="left", validate="m:1")
test_geo  = test.merge(ames,  on="PID", how="left", validate="m:1")

train_geo["GEOCODE"] = train_geo["PID"].map(pid_to_geocode10)
test_geo["GEOCODE"]  = test_geo["PID"].map(pid_to_geocode10)

miss_train = train_geo["Latitude"].isna() | train_geo["Longitude"].isna()
miss_test  = test_geo["Latitude"].isna()  | test_geo["Longitude"].isna()

missing_geocodes = pd.concat([
    train_geo.loc[miss_train, "GEOCODE"],
    test_geo.loc[miss_test, "GEOCODE"]
], ignore_index=True).dropna().unique().tolist()

print("Need ArcGIS fetch for GEOCODE count:", len(missing_geocodes))

geo_fill = fetch_centroids_by_geocode(missing_geocodes, chunk_size=200)

train_geo = train_geo.merge(geo_fill, on="GEOCODE", how="left", suffixes=("", "_arc"))
test_geo  = test_geo.merge(geo_fill,  on="GEOCODE", how="left", suffixes=("", "_arc"))

train_geo["Latitude"]  = train_geo["Latitude"].fillna(train_geo["Latitude_arc"])
train_geo["Longitude"] = train_geo["Longitude"].fillna(train_geo["Longitude_arc"])
test_geo["Latitude"]   = test_geo["Latitude"].fillna(test_geo["Latitude_arc"])
test_geo["Longitude"]  = test_geo["Longitude"].fillna(test_geo["Longitude_arc"])

train_geo = train_geo.drop(columns=["Latitude_arc","Longitude_arc"], errors="ignore")
test_geo  = test_geo.drop(columns=["Latitude_arc","Longitude_arc"], errors="ignore")

print("Train missing lat/lon after fill:", train_geo[["Latitude","Longitude"]].isna().any(axis=1).sum())
print("Test  missing lat/lon after fill :", test_geo[["Latitude","Longitude"]].isna().any(axis=1).sum())

out_train = fr"C:\Users\Admin\Downloads\train_with_PID_latlon.csv"
out_test  = fr"C:\Users\Admin\Downloads\test_with_PID_latlon.csv"

train_geo.to_csv(out_train, index=False, encoding="utf-8-sig")
test_geo.to_csv(out_test, index=False, encoding="utf-8-sig")
print("Saved:", out_train)
print("Saved:", out_test)


Need ArcGIS fetch for GEOCODE count: 446
Train missing lat/lon after fill: 13
Test  missing lat/lon after fill : 13
Saved: C:\Users\Admin\Downloads\train_with_PID_latlon_20251212_185128.csv
Saved: C:\Users\Admin\Downloads\test_with_PID_latlon_20251212_185128.csv


In [7]:
train_geo[train_geo["Latitude"].isna()][["Id","PID","GEOCODE"]].head(20)



,Id,PID,GEOCODE
46,47,914465040,0914465040
260,261,535300120,0535300120
371,372,904101170,0904101170
445,446,905450020,0905450020
479,480,902202070,0902202070
495,496,902477120,0902477120
529,530,909475070,0909475070
557,558,911175360,0911175360
583,584,902401120,0902401120
681,682,909129100,0909129100


In [8]:
test_geo[test_geo["Latitude"].isna()][["Id","PID","GEOCODE"]].head(20)


,Id,PID,GEOCODE
31,1492,531477050,0531477050
98,1559,904351040,0904351040
335,1796,535426150,0535426150
354,1815,902205010,0902205010
362,1823,902477130,0902477130
455,1916,912251110,0912251110
626,2087,902103150,0902103150
984,2445,902205020,0902205020
1133,2594,916403040,0916403040
1137,2598,916477060,0916477060
